# NLP Lab Task 5 – N-gram Language Models
Standard library only. Lowercase everything, `<s>` and `</s>` markers, V = all words + `</s>` (no `<s>`), natural log, 4 decimals.

In [1]:
import math, itertools
from collections import Counter

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test  = ["the cat sat on the rug", "the fish sat on the mat"]
r4 = lambda x: round(x, 4)

## Task 1: Tokenization

In [2]:
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split(" ") + ["</s>"]

train_tok = [tokenize(s) for s in train]
test_tok  = [tokenize(s) for s in test]
print("TRAIN"); [print(t) for t in train_tok]
print("TEST");  [print(t) for t in test_tok]

print("Tokens with markers   :", sum(len(t) for t in train_tok))
print("Tokens without markers:", sum(len(t) - 2 for t in train_tok))

def build_vocab(sentences):
    # all words plus </s>, no <s>
    vocab = set()
    for s in sentences:
        vocab.update(tokenize(s)[1:])   # skip <s>
    return vocab

vocab = build_vocab(train)
V = len(vocab)
print("V =", V)
print("Vocabulary:", sorted(vocab))

TRAIN
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
TEST
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']
Tokens with markers   : 30
Tokens without markers: 22
V = 11
Vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


## Task 2: Generating N-grams

In [3]:
def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

first = train_tok[0]
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    print(name, get_ngrams(first, n))

# verify m - n + 1 rule
for t in train_tok + test_tok:
    m = len(t)
    for n in (1, 2, 3):
        assert len(get_ngrams(t, n)) == m - n + 1
print("m - n + 1 rule verified for all sentences, n = 1, 2, 3")

print("n larger than sentence:", get_ngrams(first, 20))   # empty list, no crash

Unigrams [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
Bigrams [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
Trigrams [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]
m - n + 1 rule verified for all sentences, n = 1, 2, 3
n larger than sentence: []


## Task 3: Counting N-grams

In [4]:
uni = Counter(); bi = Counter(); tri = Counter()
for t in train_tok:
    uni.update(get_ngrams(t, 1))
    bi.update(get_ngrams(t, 2))
    tri.update(get_ngrams(t, 3))

def show(counter, title):
    print(f"--- {title} ---")
    for g, c in sorted(counter.items(), key=lambda x: (-x[1], x[0])):
        print(f"{g}: {c}")

ctx = Counter()                      # C(v) as a history (bigrams starting with v)
for (a, b), c in bi.items(): ctx[a] += c

show(uni, "Unigram counts"); show(bi, "Bigram counts"); show(tri, "Trigram counts")

print("Top 5 bigrams:", bi.most_common(5))
print("Distinct unigrams/bigrams/trigrams:", len(uni), len(bi), len(tri))

def count_of(ngram):
    n = len(ngram)
    return {1: uni, 2: bi, 3: tri}[n].get(tuple(ngram), 0)

print("seen   ('the','cat'):", count_of(("the", "cat")))
print("unseen ('the','bone'):", count_of(("the", "bone")))

--- Unigram counts ---
('the',): 8
('</s>',): 4
('<s>',): 4
('ate',): 2
('cat',): 2
('dog',): 2
('on',): 2
('sat',): 2
('bone',): 1
('fish',): 1
('mat',): 1
('rug',): 1
--- Bigram counts ---
('<s>', 'the'): 4
('ate', 'the'): 2
('on', 'the'): 2
('sat', 'on'): 2
('the', 'cat'): 2
('the', 'dog'): 2
('bone', '</s>'): 1
('cat', 'ate'): 1
('cat', 'sat'): 1
('dog', 'ate'): 1
('dog', 'sat'): 1
('fish', '</s>'): 1
('mat', '</s>'): 1
('rug', '</s>'): 1
('the', 'bone'): 1
('the', 'fish'): 1
('the', 'mat'): 1
('the', 'rug'): 1
--- Trigram counts ---
('<s>', 'the', 'cat'): 2
('<s>', 'the', 'dog'): 2
('sat', 'on', 'the'): 2
('ate', 'the', 'bone'): 1
('ate', 'the', 'fish'): 1
('cat', 'ate', 'the'): 1
('cat', 'sat', 'on'): 1
('dog', 'ate', 'the'): 1
('dog', 'sat', 'on'): 1
('on', 'the', 'mat'): 1
('on', 'the', 'rug'): 1
('the', 'bone', '</s>'): 1
('the', 'cat', 'ate'): 1
('the', 'cat', 'sat'): 1
('the', 'dog', 'ate'): 1
('the', 'dog', 'sat'): 1
('the', 'fish', '</s>'): 1
('the', 'mat', '</s>'): 1
('th

## Task 4: Unigram model
N = number of predicted tokens (all tokens except `<s>`), so probabilities sum to 1 over V.

In [5]:
N = sum(c for (w,), c in uni.items() if w != "<s>")   # 26
print("N =", N)

def unigram_prob(w):
    return uni.get((w,), 0) / N

print(f"{'word':8}{'count':>6}{'P(w)':>9}")
for w in sorted(vocab):
    print(f"{w:8}{uni[(w,)]:>6}{r4(unigram_prob(w)):>9}")
assert abs(sum(unigram_prob(w) for w in vocab) - 1) < 1e-9
print("Sum of probabilities = 1  ✓")

def unigram_sentence_prob(sentence):
    p = 1.0
    for w in tokenize(sentence)[1:]:        # words + </s>
        p *= unigram_prob(w)
    return p

for s in test:
    print(s, "->", unigram_sentence_prob(s))

words = test[0].split()
probs = {round(unigram_sentence_prob(" ".join(p)), 15) for p in itertools.permutations(words)}
print("Distinct permutations:", len(set(itertools.permutations(words))),
      "| distinct probability values:", len(probs))
print("Different word orders give the SAME unigram probability (unigrams ignore order).")

N = 26
word     count     P(w)
</s>         4   0.1538
ate          2   0.0769
bone         1   0.0385
cat          2   0.0769
dog          2   0.0769
fish         1   0.0385
mat          1   0.0385
on           2   0.0769
rug          1   0.0385
sat          2   0.0769
the          8   0.3077
Sum of probabilities = 1  ✓
the cat sat on the rug -> 2.549861058867735e-07
the fish sat on the mat -> 1.2749305294338675e-07
Distinct permutations: 360 | distinct probability values: 1
Different word orders give the SAME unigram probability (unigrams ignore order).


## Task 5: Bigram model (MLE)

In [6]:
def bigram_prob(prev, w):
    c = ctx.get(prev, 0)
    return bi.get((prev, w), 0) / c if c else 0.0

def next_word_distribution(prev):
    d = {w: bigram_prob(prev, w) for w in vocab}
    return {w: p for w, p in d.items() if p > 0}

d = next_word_distribution("the")
print({w: r4(p) for w, p in sorted(d.items())})
assert abs(sum(d.values()) - 1) < 1e-9
print("Sum = 1  ✓")

def bigram_sentence_prob(sentence):
    t = tokenize(sentence); p = 1.0
    for prev, w in get_ngrams(t, 2):
        p *= bigram_prob(prev, w)
    return p

for s in test:
    print(s, "->", bigram_sentence_prob(s))

def predict_next(prev):
    d = next_word_distribution(prev)
    return min(d, key=lambda w: (-d[w], w)) if d else None

for w in ["<s>", "the", "cat", "sat", "on", "ate", "dog"]:
    print(f"predict_next({w!r}) = {predict_next(w)!r}")

{'bone': 0.125, 'cat': 0.25, 'dog': 0.25, 'fish': 0.125, 'mat': 0.125, 'rug': 0.125}
Sum = 1  ✓
the cat sat on the rug -> 0.015625
the fish sat on the mat -> 0.0
predict_next('<s>') = 'the'
predict_next('the') = 'cat'
predict_next('cat') = 'ate'
predict_next('sat') = 'on'
predict_next('on') = 'the'
predict_next('ate') = 'the'
predict_next('dog') = 'ate'


## Task 6: Trigram model

In [7]:
def trigram_prob(w1, w2, w3):
    c = bi.get((w1, w2), 0)
    return tri.get((w1, w2, w3), 0) / c if c else 0.0

def trigram_sentence_prob(sentence):
    t = tokenize(sentence)
    p = bigram_prob(t[0], t[1])              # first word given <s>: bigram
    for w1, w2, w3 in get_ngrams(t, 3):
        p *= trigram_prob(w1, w2, w3)
    return p

for s in test:
    print(s, "->", trigram_sentence_prob(s))

print(f"\n{'N-gram':10}{'Distinct':>9}")
for name, c in [("Unigram", uni), ("Bigram", bi), ("Trigram", tri)]:
    print(f"{name:10}{len(c):>9}")

tot_b = tot_t = 0
for s in test:
    t = tokenize(s)
    ub = [g for g in get_ngrams(t, 2) if count_of(g) == 0]
    ut = [g for g in get_ngrams(t, 3) if count_of(g) == 0]
    tot_b += len(ub); tot_t += len(ut)
    print(f"\n{s}\n  unseen bigrams : {ub}\n  unseen trigrams: {ut}")
print("\nTotal unseen bigrams:", tot_b, "| unseen trigrams:", tot_t)
print("More unseen N-grams ->", "trigram" if tot_t > tot_b else "bigram", "model")

the cat sat on the rug -> 0.125
the fish sat on the mat -> 0.0

N-gram     Distinct
Unigram          12
Bigram           18
Trigram          19

the cat sat on the rug
  unseen bigrams : []
  unseen trigrams: []

the fish sat on the mat
  unseen bigrams : [('fish', 'sat')]
  unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]

Total unseen bigrams: 1 | unseen trigrams: 3
More unseen N-grams -> trigram model


## Task 7: Zero-probability problem

In [8]:
def unseen_bigrams(sentence):
    return [g for g in get_ngrams(tokenize(sentence), 2) if count_of(g) == 0]

for s in test:
    print(s, "| unseen:", unseen_bigrams(s), "| P_MLE =", bigram_sentence_prob(s))
assert bigram_sentence_prob(test[1]) == 0
print("Second test sentence has probability 0  ✓")

words_only = sorted(vocab - {"</s>"})
pairs = [(a, b) for a in words_only for b in words_only]
unseen = [p for p in pairs if bi.get(p, 0) == 0]
print(f"Unseen word pairs: {len(unseen)} / {len(pairs)} = {100*len(unseen)/len(pairs):.2f}%")

def safe_log(p):
    return float("-inf") if p == 0 else math.log(p)

print("ln P(test[1]) =", safe_log(bigram_sentence_prob(test[1])))

the cat sat on the rug | unseen: [] | P_MLE = 0.015625
the fish sat on the mat | unseen: [('fish', 'sat')] | P_MLE = 0.0
Second test sentence has probability 0  ✓
Unseen word pairs: 87 / 100 = 87.00%
ln P(test[1]) = -inf


## Task 8: Smoothing

In [9]:
def bigram_prob_addone(prev, w):
    return (bi.get((prev, w), 0) + 1) / (ctx.get(prev, 0) + V)

print(f"{'':16}{'MLE':>8}{'Add-one':>9}")
for prev, w in [("fish", "sat"), ("the", "cat")]:
    print(f"P({w} | {prev})".ljust(16), f"{r4(bigram_prob(prev, w)):>8}{r4(bigram_prob_addone(prev, w)):>9}")

for prev in sorted(vocab) + ["<s>"]:
    assert abs(sum(bigram_prob_addone(prev, w) for w in vocab) - 1) < 1e-9
print("Add-one next-word probabilities sum to 1 for every previous word  ✓")

def bigram_prob_addk(prev, w, k):
    return (bi.get((prev, w), 0) + k) / (ctx.get(prev, 0) + k * V)

print("\n  k    P(sat | fish)")
for k in [1, 0.5, 0.1, 0.01]:
    print(f"{k:<6}{r4(bigram_prob_addk('fish', 'sat', k))}")

                     MLE  Add-one
P(sat | fish)         0.0   0.0833
P(cat | the)         0.25   0.1579
Add-one next-word probabilities sum to 1 for every previous word  ✓

  k    P(sat | fish)
1     0.0833
0.5   0.0769
0.1   0.0476
0.01  0.009


## Task 9: Perplexity and model comparison
Perplexity = exp(−(1/M) Σ ln P), where M = number of predicted tokens (words + `</s>`).

In [10]:
def perplexity(sentences, prob_fn):
    """prob_fn(prev, w) -> probability. Unigram models just ignore prev."""
    total, M = 0.0, 0
    for s in sentences:
        for prev, w in get_ngrams(tokenize(s), 2):
            p = prob_fn(prev, w)
            if p == 0:
                return float("inf")
            total += math.log(p); M += 1
    return math.exp(-total / M)

uni_addone = lambda prev, w: (uni.get((w,), 0) + 1) / (N + V)

print(f"{'Model':22}{'Perplexity':>12}")
print(f"{'Bigram MLE':22}{perplexity(test, bigram_prob):>12}")
print(f"{'Bigram add-one':22}{r4(perplexity(test, bigram_prob_addone)):>12}")
print(f"{'Unigram add-one':22}{r4(perplexity(test, uni_addone)):>12}")

ks = [1, 0.5, 0.1, 0.01]
res = {k: perplexity(test, lambda p, w, k=k: bigram_prob_addk(p, w, k)) for k in ks}
print(f"\n{'k':<8}{'Perplexity':>12}")
for k in ks: print(f"{k:<8}{r4(res[k]):>12}")
best_test_k = min(res, key=res.get)
print("Best k on test:", best_test_k)

Model                   Perplexity
Bigram MLE                     inf
Bigram add-one              5.7969
Unigram add-one              9.137

k         Perplexity
1             5.7969
0.5           4.5703
0.1           2.9559
0.01          2.6272
Best k on test: 0.01


In [11]:
# Train on first 3 sentences only, V still built from all 4 sentences
train3 = train[:3]; val = [train[3]]
bi3, ctx3 = Counter(), Counter()
for s in train3:
    for a, b in get_ngrams(tokenize(s), 2):
        bi3[(a, b)] += 1; ctx3[a] += 1
V_all = len(build_vocab(train))

def addk3(prev, w, k):
    return (bi3.get((prev, w), 0) + k) / (ctx3.get(prev, 0) + k * V_all)

val_res  = {k: perplexity(val,  lambda p, w, k=k: addk3(p, w, k)) for k in ks}
test_res = {k: perplexity(test, lambda p, w, k=k: addk3(p, w, k)) for k in ks}
print(f"{'k':<8}{'Validation PP':>15}{'Test PP':>10}")
for k in ks: print(f"{k:<8}{r4(val_res[k]):>15}{r4(test_res[k]):>10}")

best_val_k  = min(val_res,  key=val_res.get)
best_test_k3 = min(test_res, key=test_res.get)
print("\nBest k on validation:", best_val_k, "-> test perplexity =", r4(test_res[best_val_k]))
print("Best k directly on test:", best_test_k3, "-> test perplexity =", r4(test_res[best_test_k3]))

k         Validation PP   Test PP
1                8.5857    5.7406
0.5              8.0163    4.4539
0.1               8.026    2.7778
0.01            13.1304    2.4258

Best k on validation: 0.5 -> test perplexity = 4.4539
Best k directly on test: 0.01 -> test perplexity = 2.4258


**Takeaway:** MLE gives zero probability (infinite perplexity) to unseen bigrams; smoothing fixes this. Choosing k on held-out validation data is the honest way to tune; picking k on the test set is optimistic (it leaks test information).